In [ ]:
# base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import warnings

# models
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor

# optimization hyperparameters
import optuna
from optuna.integration import XGBoostPruningCallback
from optuna.integration.lightgbm import LightGBMPruningCallback
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

# log
import sys
from pathlib import Path
from src.config.log_files import auto_logger, setup_log

# path
from src.config.path import PROCESS_PATH

In [ ]:
warnings.filterwarnings('ignore')

df = pd.read_parquet(PROCESS_PATH / "flight.parquet", engine="pyarrow")
df

,flight,payload,route,altitude_preset,date,time_day,duration_s,total_distance_m,wind_speed_mean,wind_speed_std,...,velocity_mag_max,acceleration_mag_mean,acceleration_mag_std,battery_voltage_mean,battery_voltage_min,max_height_agl,final_height_agl,energy_consumed_Wh,battery_needed_mAh,landing_offset_flag
0,1,0.0,R5,25,2019-04-07,10:13,200.70,549.065984,3.898058,1.952675,...,6.261616,9.842870,0.466372,22.070134,21.228519,26.257755,-0.994409,21.769975,1000.743935,False
1,2,0.0,R5,50,2019-04-07,10:23,271.20,666.615556,3.522941,2.159456,...,7.676739,9.881874,0.628406,21.527547,20.125463,52.637988,0.589477,25.366627,1205.255874,False
2,3,0.0,R5,25,2019-04-07,10:33,180.10,577.009957,4.581182,3.335733,...,7.213987,9.902090,0.545290,22.330305,19.943916,24.660462,0.106140,17.094392,789.073853,False
3,4,0.0,R5,25,2019-04-07,10:48,171.00,562.802357,4.596319,3.438072,...,9.425537,9.900368,0.559073,21.950616,20.365856,25.580572,0.416669,14.690038,687.813976,False
4,5,0.0,R2,25,2019-04-07,11:05,217.00,470.978276,3.333910,2.247522,...,4.900079,9.817243,0.341981,21.519937,18.923494,24.323036,-0.924901,19.019928,920.070980,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,275,500.0,R1,25,2019-10-24,9:05,149.40,518.231162,4.139878,3.885389,...,8.788151,9.850903,0.420902,22.056616,20.332052,21.959932,-3.264891,16.295811,760.480643,False
205,276,500.0,R1,25,2019-10-24,9:32,147.90,517.758034,4.392581,4.332293,...,10.553163,9.862172,0.471081,21.492353,19.788662,22.542598,1.293232,15.392088,738.620356,False
206,277,500.0,R1,25,2019-10-24,9:45,134.81,517.677109,5.524651,4.029744,...,10.579715,9.847796,0.529103,21.908016,19.352947,24.289124,-0.286362,15.531389,741.451254,False
207,278,500.0,R7,25-50-100-25,2019-10-24,10:00,186.39,545.413261,4.686967,3.826570,...,10.376503,9.829065,0.456918,22.394109,20.407175,95.041887,-0.457578,18.922540,887.198699,False


In [25]:
def run_hyperopt(name, obj_fn, n_trials=50):
    study = optuna.create_study(
        direction="minimize",
        study_name=f"{name}_drone_energy",
        load_if_exists=True,
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=20),
    )

    study.optimize(
        lambda trial: obj_fn(trial, X, y),
        n_trials=n_trials,
    )
    return study

In [26]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def objective_ridge(trial, X, y):
    params = {
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
    }
    scores = []
    for train_idx, val_idx in kf.split(X):
        model = Ridge(**params)
        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[val_idx])
        scores.append(np.sqrt(root_mean_squared_error(y[val_idx], preds)))
    return np.mean(scores)

def objective_ada(trial, X, y):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 1.0, log=True),
        "loss": trial.suggest_categorical("loss", ["linear", "square", "exponential"]),
        "random_state": 42,
    }
    scores = []
    for train_idx, val_idx in kf.split(X):
        model = AdaBoostRegressor(**params)
        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[val_idx])
        scores.append(np.sqrt(root_mean_squared_error(y[val_idx], preds)))
    return np.mean(scores)

def objective_rf(trial, X, y):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 2, 8),          # tightened: 209 rows
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "random_state": 42,
        "n_jobs": -1,
    }
    scores = []
    for train_idx, val_idx in kf.split(X):
        model = RandomForestRegressor(**params)
        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[val_idx])
        scores.append(np.sqrt(root_mean_squared_error(y[val_idx], preds)))
    return np.mean(scores)

def objective_xgb(trial, X, y):
    params = {
        "objective": "reg:squarederror",
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "random_state": 42,
        "n_jobs": -1,
    }
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        pruning_callback = XGBoostPruningCallback(trial, "validation_0-rmse")

        model = XGBRegressor(
            **params,
            eval_metric="rmse",
            callbacks=[pruning_callback],   # moved here — constructor, not fit()
        )
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

        preds = model.predict(X_val)
        scores.append(np.sqrt(root_mean_squared_error(y_val, preds)))
    return np.mean(scores)

def objective_lgbm(trial, X, y):
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 8, 40),       # tightened
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 3, 30),
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }
    scores = []
    for train_idx, val_idx in kf.split(X):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        pruning_callback = LightGBMPruningCallback(trial, "rmse")
        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                  eval_metric="rmse", callbacks=[pruning_callback])
        preds = model.predict(X_val)
        scores.append(np.sqrt(root_mean_squared_error(y_val, preds)))
    return np.mean(scores)

In [27]:
feature_cols = [
    'total_distance_m',
    'max_height_agl',
    'payload',
    'wind_speed_mean',
    'velocity_mag_mean',
    'speed_max',
    'speed_mean',
    'acceleration_mag_mean'
] 
target = 'duration_s'

X = df[feature_cols].values
y = df[target].values

studies = {}
objectives = {
    "ridge": objective_ridge,
    "ada": objective_ada,
    "rf": objective_rf,
    "xgb": objective_xgb,
    "lgbm": objective_lgbm,
}

for name, obj_fn in objectives.items():
    study = run_hyperopt(name, obj_fn, n_trials=50)
    studies[name] = study
    print(f"{name} best RMSE: {study.best_value:.4f} | params: {study.best_params}")

[I 2026-07-16 00:49:11,551] A new study created in memory with name: ridge_drone_energy
[I 2026-07-16 00:49:11,566] Trial 0 finished with value: 3.7211293130372396 and parameters: {'alpha': 0.07054241537006523}. Best is trial 0 with value: 3.7211293130372396.
[I 2026-07-16 00:49:11,572] Trial 1 finished with value: 3.7390761858724106 and parameters: {'alpha': 1.2647035370422286}. Best is trial 0 with value: 3.7211293130372396.
[I 2026-07-16 00:49:11,578] Trial 2 finished with value: 3.723894414231738 and parameters: {'alpha': 0.3672615150261462}. Best is trial 0 with value: 3.7211293130372396.
[I 2026-07-16 00:49:11,584] Trial 3 finished with value: 3.7241097871472193 and parameters: {'alpha': 0.004659407210123157}. Best is trial 0 with value: 3.7211293130372396.
[I 2026-07-16 00:49:11,589] Trial 4 finished with value: 3.7231901148870663 and parameters: {'alpha': 0.009305031939892522}. Best is trial 0 with value: 3.7211293130372396.
[I 2026-07-16 00:49:11,596] Trial 5 finished with val

ridge best RMSE: 3.7211 | params: {'alpha': 0.07519487100334402}


[I 2026-07-16 00:49:12,765] Trial 0 finished with value: 4.526492735606549 and parameters: {'n_estimators': 181, 'learning_rate': 0.018543359761775992, 'loss': 'linear'}. Best is trial 0 with value: 4.526492735606549.
[I 2026-07-16 00:49:13,700] Trial 1 finished with value: 4.3488870881548705 and parameters: {'n_estimators': 209, 'learning_rate': 0.7563702921860803, 'loss': 'exponential'}. Best is trial 1 with value: 4.3488870881548705.
[I 2026-07-16 00:49:15,444] Trial 2 finished with value: 4.484290730298396 and parameters: {'n_estimators': 378, 'learning_rate': 0.023053573149930243, 'loss': 'square'}. Best is trial 1 with value: 4.3488870881548705.
[I 2026-07-16 00:49:17,248] Trial 3 finished with value: 4.399934052104069 and parameters: {'n_estimators': 403, 'learning_rate': 0.03971809260586794, 'loss': 'linear'}. Best is trial 1 with value: 4.3488870881548705.
[I 2026-07-16 00:49:18,465] Trial 4 finished with value: 4.4627871080929795 and parameters: {'n_estimators': 271, 'learnin

ada best RMSE: 4.2720 | params: {'n_estimators': 393, 'learning_rate': 0.4522723702483474, 'loss': 'square'}


[I 2026-07-16 00:50:27,572] Trial 0 finished with value: 4.827649358797286 and parameters: {'n_estimators': 268, 'max_depth': 3, 'min_samples_split': 13, 'min_samples_leaf': 10, 'max_features': 'log2'}. Best is trial 0 with value: 4.827649358797286.
[I 2026-07-16 00:50:29,590] Trial 1 finished with value: 4.260874173908528 and parameters: {'n_estimators': 461, 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': None}. Best is trial 1 with value: 4.260874173908528.
[I 2026-07-16 00:50:31,252] Trial 2 finished with value: 4.306575244790405 and parameters: {'n_estimators': 381, 'max_depth': 4, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 1 with value: 4.260874173908528.
[I 2026-07-16 00:50:32,993] Trial 3 finished with value: 4.301060246012532 and parameters: {'n_estimators': 394, 'max_depth': 3, 'min_samples_split': 6, 'min_samples_leaf': 6, 'max_features': None}. Best is trial 1 with value: 4.260874173908528.
[I 2026-0

rf best RMSE: 3.8886 | params: {'n_estimators': 412, 'max_depth': 8, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': None}


[I 2026-07-16 00:51:21,579] Trial 0 finished with value: 4.0141371958041985 and parameters: {'max_depth': 6, 'learning_rate': 0.1121601253483441, 'subsample': 0.7543490488443351, 'colsample_bytree': 0.5718763616512232, 'reg_alpha': 0.4833268908231422, 'reg_lambda': 3.1647672657891315, 'min_child_weight': 8, 'n_estimators': 438}. Best is trial 0 with value: 4.0141371958041985.
[I 2026-07-16 00:51:22,171] Trial 1 finished with value: 3.9492937311510885 and parameters: {'max_depth': 6, 'learning_rate': 0.039948161301235775, 'subsample': 0.7102997583675119, 'colsample_bytree': 0.7426361432642832, 'reg_alpha': 0.6515012166725463, 'reg_lambda': 0.0044862689397331705, 'min_child_weight': 1, 'n_estimators': 242}. Best is trial 1 with value: 3.9492937311510885.
[I 2026-07-16 00:51:22,555] Trial 2 finished with value: 4.1825733662112565 and parameters: {'max_depth': 3, 'learning_rate': 0.014024551944121435, 'subsample': 0.5141847825969559, 'colsample_bytree': 0.5919221081798143, 'reg_alpha': 8.7

xgb best RMSE: 3.5815 | params: {'max_depth': 3, 'learning_rate': 0.13134475757064812, 'subsample': 0.838706297390557, 'colsample_bytree': 0.9009396859862594, 'reg_alpha': 0.0031231223121617247, 'reg_lambda': 0.032877834020447624, 'min_child_weight': 3, 'n_estimators': 216}


[I 2026-07-16 00:51:31,719] Trial 1 finished with value: 4.600573381492149 and parameters: {'num_leaves': 29, 'max_depth': 2, 'learning_rate': 0.12030692166209155, 'feature_fraction': 0.924544054778186, 'bagging_fraction': 0.9906110465246216, 'min_child_samples': 18, 'n_estimators': 461}. Best is trial 0 with value: 4.365173264999947.
[I 2026-07-16 00:51:32,115] Trial 2 finished with value: 4.548875196994983 and parameters: {'num_leaves': 12, 'max_depth': 4, 'learning_rate': 0.09110496380691915, 'feature_fraction': 0.8774282431309375, 'bagging_fraction': 0.6201308071826052, 'min_child_samples': 23, 'n_estimators': 382}. Best is trial 0 with value: 4.365173264999947.
[I 2026-07-16 00:51:32,308] Trial 3 finished with value: 3.732301496625046 and parameters: {'num_leaves': 9, 'max_depth': 7, 'learning_rate': 0.04730150495591336, 'feature_fraction': 0.9980765069450328, 'bagging_fraction': 0.704957001714468, 'min_child_samples': 9, 'n_estimators': 206}. Best is trial 3 with value: 3.7323014

lgbm best RMSE: 3.5219 | params: {'num_leaves': 37, 'max_depth': 4, 'learning_rate': 0.06917366736408287, 'feature_fraction': 0.8008892283061073, 'bagging_fraction': 0.5365330336088984, 'min_child_samples': 6, 'n_estimators': 147}


In [ ]:
ridge = Ridge(
    alpha=0.07519487100334402,
    random_state=42,
)

ada = AdaBoostRegressor(
    n_estimators=393,
    learning_rate=0.4522723702483474,
    loss="square",
    random_state=42,
)

rf = RandomForestRegressor(
    n_estimators=412,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features=None,
    n_jobs=-1,
    random_state=42,
)

xgb = XGBRegressor(
    max_depth=3,
    learning_rate=0.13134475757064812,
    subsample=0.838706297390557,
    colsample_bytree=0.9009396859862594,
    reg_alpha=0.0031231223121617247,
    reg_lambda=0.032877834020447624,
    min_child_weight=3,
    n_estimators=216,
    n_jobs=-1,
    random_state=42,
)

lgbm = LGBMRegressor(
    num_leaves=37,
    max_depth=4,
    learning_rate=0.06917366736408287,
    feature_fraction=0.8008892283061073,
    bagging_fraction=0.5365330336088984,
    min_child_samples=6,
    n_estimators=147,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)

,num_leaves,8
,max_depth,2
,learning_rate,0.21685220605980327
,n_estimators,384
,min_child_samples,3
,random_state,42
,n_jobs,-1
,feature_fraction,0.8029048955724699
,bagginh_fraction,0.6529537513693758
,boosting_type,'gbdt'
,subsample_for_bin,200000


In [21]:
del df
gc.collect()
%whos

Variable                  Type         Data/Info
------------------------------------------------
AdaBoostRegressor         ABCMeta      <class 'sklearn.ensemble.<...>sting.AdaBoostRegressor'>
KFold                     ABCMeta      <class 'sklearn.model_selection._split.KFold'>
LGBMRegressor             type         <class 'lightgbm.sklearn.LGBMRegressor'>
LightGBMPruningCallback   type         <class 'optuna_integratio<...>LightGBMPruningCallback'>
Path                      type         <class 'pathlib.Path'>
RandomForestRegressor     ABCMeta      <class 'sklearn.ensemble.<...>t.RandomForestRegressor'>
Ridge                     ABCMeta      <class 'sklearn.linear_model._ridge.Ridge'>
XGBRegressor              type         <class 'xgboost.sklearn.XGBRegressor'>
XGBoostPruningCallback    ABCMeta      <class 'optuna_integratio<...>.XGBoostPruningCallback'>
auto_logger               function     <function auto_logger at 0x7fd443341800>
feature_cols              list         n=8
gc        